# AAPL Machine Learning Trading Model: Quantitative Time-Series Classification
### Single-Stock Modeling with Adjustable Horizons, Return Thresholds & Realistic Backtesting

This Jupyter Notebook implements a machine learning trading model for **Apple Inc. (NASDAQ: AAPL)** following the project requirements defined in `Machine Learning Project_ S&P 500 Trading Model.docx`.

#### Key Architectural Principles:
1. **Zero Data Leakage:** All technical features are strictly calculated on historical data available up to time t. Targets look ahead N days, but features never look ahead.
2. **Adjustable Target Parameters:** Future prediction horizon (N days) and return thresholds (Buy, Sell) are fully configurable.
3. **Chronological Splitting:** Training, validation, and test datasets maintain temporal integrity without random shuffling.
4. **Realistic Backtesting:** Trades execute on day t+1, incorporating a 0.1% fee drag per transaction.


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, precision_score, recall_score, f1_score, roc_auc_score

# Plotting settings
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10


## 1. Configurable Model & Trading Parameters
You can adjust the future look-ahead period (`FUTURE_PERIOD`) and return thresholds (`BUY_THRESHOLD`, `SELL_THRESHOLD`) below to experiment with different trading horizons and signal sensitivities.


In [ ]:
# ==========================================
# ADJUSTABLE PROJECT PARAMETERS FOR AAPL
# ==========================================
TICKER = 'AAPL'
FUTURE_PERIOD = 5          # N-day look-ahead prediction horizon (trading days)
BUY_THRESHOLD = 0.015      # +1.5% return threshold over FUTURE_PERIOD to classify as BUY (1)
SELL_THRESHOLD = -0.015    # -1.5% return threshold over FUTURE_PERIOD to classify as SELL (-1)

# Backtest Configuration
INITIAL_CAPITAL = 100000.0  # $100,000 portfolio base
TRANSACTION_FEE = 0.001     # 0.1% cost per trade (slippage + broker fee)
CONFIDENCE_THRESHOLD = 0.45 # Minimum probability required to trigger non-Hold signal

print(f"=== Model Configured for {TICKER} ===")
print(f"Horizon: {FUTURE_PERIOD} trading days")
print(f"Buy Threshold: > +{BUY_THRESHOLD*100:.1f}%")
print(f"Sell Threshold: < {SELL_THRESHOLD*100:.1f}%")
print(f"Transaction Fee Drag: {TRANSACTION_FEE*100:.1f}% per trade")


## 2. Data Retrieval Pipeline
Fetches historical daily OHLCV data for AAPL from 2012 to 2024. Includes a deterministic synthetic fallback generator in offline sandbox environments.


In [ ]:
def load_aapl_data():
    try:
        import yfinance as yf
        print(f"Fetching historical OHLCV data for {TICKER} via yfinance...")
        df = yf.download(TICKER, start='2012-01-01', end='2024-01-01', progress=False)
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = df.columns.get_level_values(0)
        df = df[['Open', 'High', 'Low', 'Close', 'Volume']].dropna()
    except Exception as e:
        print(f"Using synthetic offline AAPL dataset (Reason: {e})")
        np.random.seed(42)
        dates = pd.date_range(start='2012-01-01', end='2024-01-01', freq='B')
        n = len(dates)
        
        # Simulate AAPL stock price dynamic with drift and volatility
        returns = np.random.normal(0.0008, 0.018, n)  # Annualized ~20% growth, ~28% vol
        price = 20.0 * np.exp(np.cumsum(returns))
        
        high = price * (1 + np.abs(np.random.normal(0, 0.008, n)))
        low = price * (1 - np.abs(np.random.normal(0, 0.008, n)))
        open_p = low + (high - low) * np.random.uniform(0.1, 0.9, n)
        volume = np.random.randint(40000000, 120000000, n)
        
        df = pd.DataFrame({
            'Open': open_p,
            'High': high,
            'Low': low,
            'Close': price,
            'Volume': volume
        }, index=dates)
    
    return df

df_raw = load_aapl_data()
print(f"Loaded {len(df_raw)} daily price bars for {TICKER} from {df_raw.index[0].strftime('%Y-%m-%d')} to {df_raw.index[-1].strftime('%Y-%m-%d')}.")


## 3. Feature Engineering Pipeline (Zero Data Leakage)
All technical indicators are computed strictly using historical price and volume up to time t.


In [ ]:
def build_features(df):
    data = df.copy()
    
    # 1. Price Returns
    data['Return_1D'] = data['Close'].pct_change(1)
    data['Return_5D'] = data['Close'].pct_change(5)
    data['Return_10D'] = data['Close'].pct_change(10)
    data['Return_21D'] = data['Close'].pct_change(21)
    
    # 2. Moving Averages Ratios
    data['SMA_10'] = data['Close'].rolling(10).mean()
    data['SMA_50'] = data['Close'].rolling(50).mean()
    data['SMA_200'] = data['Close'].rolling(200).mean()
    
    data['Ratio_SMA10'] = data['Close'] / data['SMA_10']
    data['Ratio_SMA50'] = data['Close'] / data['SMA_50']
    data['Ratio_SMA200'] = data['Close'] / data['SMA_200']
    
    # 3. Relative Strength Index (RSI_14)
    delta = data['Close'].diff()
    gain = delta.clip(lower=0).rolling(14).mean()
    loss = (-delta.clip(upper=0)).rolling(14).mean()
    rs = gain / (loss + 1e-8)
    data['RSI_14'] = 100 - (100 / (1 + rs))
    
    # 4. MACD Indicator
    ema12 = data['Close'].ewm(span=12, adjust=False).mean()
    ema26 = data['Close'].ewm(span=26, adjust=False).mean()
    data['MACD'] = ema12 - ema26
    data['MACD_Signal'] = data['MACD'].ewm(span=9, adjust=False).mean()
    data['MACD_Hist'] = data['MACD'] - data['MACD_Signal']
    
    # 5. Volatility Features (Bollinger Bands & Rolling Vol)
    data['Vol_20D'] = data['Return_1D'].rolling(20).std()
    data['BB_Middle'] = data['Close'].rolling(20).mean()
    data['BB_Std'] = data['Close'].rolling(20).std()
    data['BB_Upper'] = data['BB_Middle'] + 2 * data['BB_Std']
    data['BB_Lower'] = data['BB_Middle'] - 2 * data['BB_Std']
    data['BB_PctB'] = (data['Close'] - data['BB_Lower']) / (data['BB_Upper'] - data['BB_Lower'] + 1e-8)
    
    # 6. Volume Ratio
    data['Vol_Ratio'] = data['Volume'] / (data['Volume'].rolling(20).mean() + 1e-8)
    
    return data

df_featured = build_features(df_raw)
print("Features created successfully.")


## 4. Target Construction with Parameterized Horizon & Thresholds
Calculates the future N-day return:
$$R_{t+N} = \frac{\text{Close}_{t+N} - \text{Close}_t}{\text{Close}_t}$$

And maps it to discrete target labels:
* **Buy (1):** $R_{t+N} > \text{BUY\_THRESHOLD}$
* **Sell (-1):** $R_{t+N} < \text{SELL\_THRESHOLD}$
* **Hold (0):** Otherwise


In [ ]:
def create_target(df, horizon=FUTURE_PERIOD, buy_thresh=BUY_THRESHOLD, sell_thresh=SELL_THRESHOLD):
    data = df.copy()
    
    # Calculate future return N days ahead
    data['Future_Return'] = data['Close'].shift(-horizon) / data['Close'] - 1.0
    
    # Categorize into Buy (1), Sell (-1), Hold (0)
    conditions = [
        data['Future_Return'] > buy_thresh,
        data['Future_Return'] < sell_thresh
    ]
    choices = [1, -1]
    data['Target'] = np.select(conditions, choices, default=0)
    
    # Drop rows at the end where target cannot be computed
    data = data.dropna(subset=['Target', 'Future_Return'])
    return data

df_model = create_target(df_featured, horizon=FUTURE_PERIOD, buy_thresh=BUY_THRESHOLD, sell_thresh=SELL_THRESHOLD)
df_model = df_model.dropna() # Remove initial rolling lookback NaNs

feature_cols = [
    'Return_1D', 'Return_5D', 'Return_10D', 'Return_21D',
    'Ratio_SMA10', 'Ratio_SMA50', 'Ratio_SMA200',
    'RSI_14', 'MACD', 'MACD_Signal', 'MACD_Hist',
    'Vol_20D', 'BB_PctB', 'Vol_Ratio'
]

print(f"Dataset ready for modeling. Total samples: {len(df_model)}")
print("
Target Class Distribution:")
print(df_model['Target'].value_counts(normalize=True).rename({1: 'Buy (1)', 0: 'Hold (0)', -1: 'Sell (-1)'}).map("{:.2%}".format))


## 5. Exploratory Data Analysis (EDA)
Visualizes target class distribution and feature correlation with upcoming returns.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Plot 1: Target Class Balance
class_counts = df_model['Target'].value_counts().sort_index()
class_labels = ['Sell (-1)', 'Hold (0)', 'Buy (1)']
colors = ['#e74c3c', '#95a5a6', '#2ecc71']
axes[0].bar(class_labels, class_counts, color=colors, edgecolor='black', alpha=0.8)
axes[0].set_title(f'AAPL Target Distribution ({FUTURE_PERIOD}-Day Horizon)', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Bar Count')

# Plot 2: Top Feature Correlations with Future Returns
corr = df_model[feature_cols + ['Future_Return']].corr()['Future_Return'].drop('Future_Return').sort_values()
corr.plot(kind='barh', ax=axes[1], color='#3498db', edgecolor='black', alpha=0.8)
axes[1].set_title('Feature Correlation with Future Return', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Pearson Correlation')

plt.tight_layout()
plt.show()


## 6. Chronological Dataset Splitting
Maintains temporal ordering without data leakage across time windows:
* **Train Set:** 2012 - 2020
* **Validation Set:** 2021 - 2022
* **Out-of-Sample Test Set:** 2023 - 2024


In [ ]:
train_mask = df_model.index < '2021-01-01'
val_mask = (df_model.index >= '2021-01-01') & (df_model.index < '2023-01-01')
test_mask = df_model.index >= '2023-01-01'

X_train, y_train = df_model.loc[train_mask, feature_cols], df_model.loc[train_mask, 'Target']
X_val, y_val     = df_model.loc[val_mask, feature_cols], df_model.loc[val_mask, 'Target']
X_test, y_test   = df_model.loc[test_mask, feature_cols], df_model.loc[test_mask, 'Target']

print(f"Train samples:      {len(X_train)} ({df_model.index[train_mask][0].strftime('%Y-%m-%d')} to {df_model.index[train_mask][-1].strftime('%Y-%m-%d')})")
print(f"Validation samples: {len(X_val)} ({df_model.index[val_mask][0].strftime('%Y-%m-%d')} to {df_model.index[val_mask][-1].strftime('%Y-%m-%d')})")
print(f"Test samples:       {len(X_test)} ({df_model.index[test_mask][0].strftime('%Y-%m-%d')} to {df_model.index[test_mask][-1].strftime('%Y-%m-%d')})")


## 7. Model Training & Probability Thresholding
Trains a regularized Random Forest model with constrained tree depth (`max_depth=4`) to prevent financial noise overfitting. Uses probability thresholding (`predict_proba`) to improve trade signal precision.


In [ ]:
# Train Constrained Random Forest Model
rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=4,
    min_samples_leaf=25,
    class_weight='balanced_subsample',
    random_state=42
)

rf_model.fit(X_train, y_train)

# Generate Probabilities on Out-of-Sample Test Set
test_probs = rf_model.predict_proba(X_test)
classes = rf_model.classes_

sell_idx = np.where(classes == -1)[0][0]
buy_idx  = np.where(classes == 1)[0][0]

# Apply Custom Probability Thresholding
custom_preds = np.zeros(len(X_test), dtype=int)
for i in range(len(X_test)):
    p_buy = test_probs[i, buy_idx]
    p_sell = test_probs[i, sell_idx]
    
    if p_buy >= CONFIDENCE_THRESHOLD and p_buy > p_sell:
        custom_preds[i] = 1
    elif p_sell >= CONFIDENCE_THRESHOLD and p_sell > p_buy:
        custom_preds[i] = -1

print("=== Out-of-Sample Test Classification Report (Random Forest + Thresholding) ===")
print(classification_report(y_test, custom_preds, target_names=['Sell (-1)', 'Hold (0)', 'Buy (1)'], zero_division=0))


## 8. Realistic Backtest Engine
Executes trades on day t+1 close using signals generated at day t, accounting for a 0.1% transaction fee per position change.


In [ ]:
test_df = df_model.loc[test_mask].copy()
test_df['Signal'] = custom_preds

# Shift signal by 1 day to ensure execution at t+1 close (zero execution leakage)
test_df['Executed_Position'] = test_df['Signal'].shift(1).fillna(0)

# Calculate Daily Stock Return
test_df['Daily_Stock_Return'] = test_df['Close'].pct_change().fillna(0)

# Identify trade position changes (turnover)
test_df['Position_Change'] = test_df['Executed_Position'].diff().abs().fillna(0)
test_df['Fee_Cost'] = test_df['Position_Change'] * TRANSACTION_FEE

# Net Strategy Daily Return
test_df['Strategy_Return'] = (test_df['Executed_Position'] * test_df['Daily_Stock_Return']) - test_df['Fee_Cost']

# Cumulative Equity Curves
test_df['Strategy_Equity'] = INITIAL_CAPITAL * (1 + test_df['Strategy_Return']).cumprod()
test_df['AAPL_Equity'] = INITIAL_CAPITAL * (1 + test_df['Daily_Stock_Return']).cumprod()

print("Backtest engine complete.")


## 9. Strategy Backtest Performance & Industry Financial Metrics
Visualizes net portfolio equity vs AAPL Buy & Hold baseline and computes Sharpe, Sortino, CAGR, and Maximum Drawdown.


In [ ]:
def compute_financial_metrics(equity_curve, daily_returns, risk_free_rate=0.02):
    total_days = len(equity_curve)
    cagr = (equity_curve.iloc[-1] / equity_curve.iloc[0]) ** (252 / total_days) - 1.0
    
    ann_vol = daily_returns.std() * np.sqrt(252)
    rf_daily = risk_free_rate / 252
    
    excess_returns = daily_returns - rf_daily
    sharpe_ratio = (excess_returns.mean() / (daily_returns.std() + 1e-8)) * np.sqrt(252)
    
    downside_std = daily_returns[daily_returns < 0].std() * np.sqrt(252)
    sortino_ratio = (excess_returns.mean() * 252) / (downside_std + 1e-8)
    
    # Maximum Drawdown calculation
    rolling_peak = equity_curve.cummax()
    drawdown = (equity_curve - rolling_peak) / rolling_peak
    max_drawdown = drawdown.min()
    
    return cagr, ann_vol, sharpe_ratio, sortino_ratio, max_drawdown

strat_cagr, strat_vol, strat_sharpe, strat_sortino, strat_mdd = compute_financial_metrics(test_df['Strategy_Equity'], test_df['Strategy_Return'])
aapl_cagr, aapl_vol, aapl_sharpe, aapl_sortino, aapl_mdd = compute_financial_metrics(test_df['AAPL_Equity'], test_df['Daily_Stock_Return'])

# Plot Backtest Results
fig, ax = plt.subplots(figsize=(14, 7))
ax.plot(test_df.index, test_df['Strategy_Equity'], label=f'ML Trading Strategy (Sharpe: {strat_sharpe:.2f})', color='#2980b9', linewidth=2)
ax.plot(test_df.index, test_df['AAPL_Equity'], label=f'AAPL Buy & Hold (Sharpe: {aapl_sharpe:.2f})', color='#7f8c8d', linestyle='--', linewidth=1.5)
ax.set_title(f'Out-of-Sample Backtest: AAPL ML Strategy vs Buy & Hold (With {TRANSACTION_FEE*100:.1f}% Fee Drag)', fontsize=14, fontweight='bold')
ax.set_ylabel('Portfolio Equity ($)')
ax.legend(loc='upper left', fontsize=11)
plt.tight_layout()
plt.show()

# Print Comparison Table
metrics_df = pd.DataFrame({
    'Metric': ['CAGR', 'Annual Volatility', 'Sharpe Ratio', 'Sortino Ratio', 'Max Drawdown'],
    'ML Strategy': [f"{strat_cagr:.2%}", f"{strat_vol:.2%}", f"{strat_sharpe:.2f}", f"{strat_sortino:.2f}", f"{strat_mdd:.2%}"],
    'AAPL Buy & Hold': [f"{aapl_cagr:.2%}", f"{aapl_vol:.2%}", f"{aapl_sharpe:.2f}", f"{aapl_sortino:.2f}", f"{aapl_mdd:.2%}"]
})
print("
=== Quantitative Financial Performance Comparison ===")
print(metrics_df.to_string(index=False))


## 10. Lessons Learned & Methodological Discussion

1. **Single-Stock Volatility vs Index Noise:** Predicting directional movement for a single growth stock like Apple (NASDAQ: AAPL) presents higher idiosyncratic volatility than the broad S&P 500 index.
2. **Transaction Fee Drag:** Frequent switching between Buy and Sell signals erodes profit through 0.1% transaction costs and slippage. Probability thresholding helps preserve capital by forcing a **Hold** default when directional confidence is low.
3. **Evaluating Equity Curves:** Evaluating models solely on raw classification accuracy can be misleading. A model with moderate accuracy (40-45%) can achieve superior risk-adjusted performance (Sharpe Ratio) if it successfully limits downside drawdowns during sharp sell-offs.
